# MediMate 알약 임베딩 모델 학습 (Google Colab)

식약처 참조 크롭(25,439품목 × 앞·뒤)과 AI Hub 실제 사진 크롭으로 알약 외형 임베딩(EfficientNet + ArcFace)을 학습하고, 검색 색인을 만든다.

**준비 (PC에서)**
1. `python scripts/pack_for_colab.py` → `colab/pill-classifier-code.zip`, `colab/pill-classifier-data.zip`
2. 두 파일을 Google Drive의 `MediMate/` 폴더에 올린다 (데이터 zip은 약 1.6GB).
3. 이 노트북을 Colab에서 열고 **런타임 → 런타임 유형 변경 → T4 GPU**.

**실행**: **런타임 → 모두 실행 (Ctrl+F9)** 한 번이면 된다. Drive 권한 창이 뜨면 허용. 학습(20에폭)은 T4에서 1~2시간.
Drive에 이미 학습된 모델(`MediMate/models/embedder.pt`)이 있으면 학습은 건너뛰고 색인만 만든다. 다시 학습하려면 아래 설정 셀에서 `RETRAIN = True`.
런타임이 중간에 끊겨도 다시 **모두 실행**하면 된다. 끝난 단계는 건너뛴다.

**결과**: Drive의 `MediMate/models/embedder.onnx`(+ `embedder.pt`), `MediMate/models/index.npz`. PC의 `models/embedder.onnx`, `data/index.npz`로 복사하면 `identify.py`와 추론 서버가 자동으로 쓴다.


In [ ]:
# 0. 설정
RETRAIN = False      # True면 Drive에 모델이 있어도 다시 학습한다
EPOCHS = 20
BATCH_SIZE = 128     # 메모리 부족이면 64
DRIVE = "/content/drive/MyDrive/MediMate"
# 일반화 실험: AI Hub 약 중 이 비율을 학습에서 완전히 빼고(처음 보는 약) 평가한다. 0이면 전부 학습(서비스용 모델).
# 0.3으로 두면 결과 파일 이름에 _holdout 이 붙어 서비스용 모델과 섞이지 않는다.
HOLDOUT_FRAC = 0.0
SUFFIX = "_holdout" if HOLDOUT_FRAC > 0 else ""


In [ ]:
# 1. GPU 확인
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — 런타임 유형을 GPU로 바꾸세요")

In [ ]:
# 2. Drive 연결 (권한 창이 뜨면 허용)
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists(f"{DRIVE}/pill-classifier-code.zip"), "Drive의 MediMate/ 폴더에 pill-classifier-code.zip 이 없습니다"
assert os.path.exists(f"{DRIVE}/pill-classifier-data.zip"), "Drive의 MediMate/ 폴더에 pill-classifier-data.zip 이 없습니다"
os.makedirs(f"{DRIVE}/models", exist_ok=True)
MODEL, INDEX = f"{DRIVE}/models/embedder{SUFFIX}.pt", f"{DRIVE}/models/index{SUFFIX}.npz"
print("학습된 모델 있음:", os.path.exists(MODEL), "| 색인 있음:", os.path.exists(INDEX))


In [ ]:
# 3. 코드·데이터 풀기. 코드는 매번 Drive의 zip으로 새로 덮어쓰고(작음), 데이터는 이미 풀려 있으면 건너뛴다
import os
%cd /content
!mkdir -p pill-classifier
!unzip -o -q "{DRIVE}/pill-classifier-code.zip" -d pill-classifier
if not os.path.exists("/content/pill-classifier/data/reference/manifest.csv"):
    !unzip -q "{DRIVE}/pill-classifier-data.zip" -d pill-classifier
%cd /content/pill-classifier
!pip -q install timm rapidfuzz
!wc -l data/reference/manifest.csv data/aihub/manifest.csv


In [ ]:
# 4. 학습용 매니페스트. 평가에 쓰는 AI Hub 사진을 빼고(같은 사진으로 학습·평가하면 수치가 부풀려진다),
#    HOLDOUT_FRAC > 0 이면 약 일부를 통째로 빼서 "처음 보는 약"에 대한 효과를 잴 수 있게 한다.
import csv, random
from pathlib import Path

held_out_photos = set()
for labels in Path("data/aihub").glob("eval_photos*/labels.csv"):
    for r in csv.DictReader(open(labels, encoding="utf-8")):
        for col in ("front_src", "back_src"):
            if r.get(col):
                held_out_photos.add(Path(r[col]).stem)
rows = list(csv.DictReader(open("data/aihub/manifest.csv", encoding="utf-8")))
drugs = sorted({r["label"] for r in rows})
held_out_drugs = set(random.Random(0).sample(drugs, int(len(drugs) * HOLDOUT_FRAC))) if HOLDOUT_FRAC > 0 else set()
kept = [r for r in rows
        if r["label"] not in held_out_drugs
        and not any(Path(r["path"].replace("\\", "/")).stem.startswith(s) for s in held_out_photos)]
with open("data/aihub/manifest_train.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["path", "label", "side"]); w.writeheader(); w.writerows(kept)
Path(f"{DRIVE}/models/holdout_drugs{SUFFIX}.txt").write_text("\n".join(sorted(held_out_drugs)), encoding="utf-8")
print(f"AI Hub 크롭 {len(rows):,}장 → 학습 {len(kept):,}장 (평가 사진 제외, 통째로 뺀 약 {len(held_out_drugs)}종)")


In [ ]:
# 5. 학습. Drive에 모델이 있고 RETRAIN=False면 건너뛴다. 가장 좋은 에폭의 모델이 Drive에 바로 저장된다.
%cd /content/pill-classifier
import os
if os.path.exists(MODEL) and not RETRAIN:
    print("학습된 모델이 이미 있어 건너뜁니다:", MODEL)
else:
    !python scripts/train_embedder.py \n        --manifest data/reference/manifest.csv data/aihub/manifest_train.csv \n        --out "{MODEL}" --backbone efficientnet_b0 --img-size 224 \n        --epochs {EPOCHS} --batch-size {BATCH_SIZE} --workers 2 --eval-every 2


In [ ]:
# 6. 검색 색인 생성 (참조 크롭 + AI Hub 학습 크롭). 모델보다 오래된 색인이 있으면 다시 만든다.
%cd /content/pill-classifier
import os
assert os.path.exists(MODEL), "모델이 없습니다. 5번 셀(학습)이 끝났는지 확인하세요"
if os.path.exists(INDEX) and os.path.getmtime(INDEX) >= os.path.getmtime(MODEL):
    print("최신 색인이 이미 있어 건너뜁니다:", INDEX)
else:
    !python scripts/build_index.py --model "{MODEL}" \n        --manifest data/reference/manifest.csv data/aihub/manifest_train.csv \n        --out "{INDEX}" --batch-size 256 --workers 2
!ls -la "{DRIVE}/models"


In [ ]:
# 7. ONNX 내보내기. PC에서 PyTorch가 차단돼도(Windows Smart App Control) onnxruntime만으로 추론할 수 있다.
%cd /content/pill-classifier
import os
ONNX = f"{DRIVE}/models/embedder{SUFFIX}.onnx"
if os.path.exists(ONNX) and os.path.getmtime(ONNX) >= os.path.getmtime(MODEL):
    print("최신 ONNX가 이미 있어 건너뜁니다:", ONNX)
else:
    !pip -q install onnx onnxscript
    !python scripts/export_onnx.py --model "{MODEL}" --out "{ONNX}"
!ls -la "{DRIVE}/models"
print("완료. Drive의 MediMate/models/embedder.onnx 와 index.npz 를 PC로 내려받으세요 (embedder.pt는 PyTorch가 되는 PC에서만 필요).")


## 8. PC에서 쓰기

Drive의 `MediMate/models/embedder.onnx` → PC의 `pill-classifier/models/embedder.onnx`,
`MediMate/models/index.npz` → PC의 `pill-classifier/data/index.npz` 로 복사한다.
(`embedder.pt`는 PyTorch가 되는 PC에서만 필요하다. 추론은 .onnx로 충분하다.)

그다음 PC에서 실제 사진으로 효과를 확인한다 (임베딩이 있을 때의 가중치는 아직 조정 전이다):

```
python scripts/eval_photos.py data/aihub/eval_photos4                 # 임베딩 포함
python scripts/eval_photos.py data/aihub/eval_photos4 --no-embedding  # 비교용
```

서버(`run_server.bat`)는 두 파일이 있으면 자동으로 임베딩을 켠다 (`/health`의 `embedding: true`).
